# Simple: Multi-Valued Field (MVF) Analysis with PAMOLA.CORE

**Goal**: Learn MVF analysis basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze multi-valued fields using execute()
- Compare value distributions and combinations
- Understand field complexity metrics

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 01_mvf_analyzer_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.mvf import MVFOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from `examples/data_examples/sample.csv`
- Auto-creates sample data with multi-valued fields if file missing
- Review dataset structure before proceeding

**What you'll see:**
- Dataset summary (20 records, 3 columns expected)
- First 5 records with tags and skills arrays
- Column statistics (types, unique values)
- Multi-valued fields: tags (list format), skills (JSON format)

**Note:** Sample demonstrates both list-style ['item1', 'item2'] and JSON-style arrays

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with multi-valued fields
    sample_data = pd.DataFrame({
        'record_id': range(1, 21),
        'tags': [
            "['python', 'data science', 'machine learning']",
            "['java', 'backend', 'api']",
            "['python', 'web development']",
            "['data science', 'statistics', 'r']",
            "['machine learning', 'deep learning', 'ai']",
            "['javascript', 'frontend', 'react']",
            "['python', 'automation']",
            "['devops', 'kubernetes', 'docker']",
            "['data science', 'python', 'visualization']",
            "['backend', 'nodejs', 'api']",
            "['machine learning', 'python']",
            "['frontend', 'vue', 'css']",
            "['cloud', 'aws', 'azure']",
            "['python', 'django', 'web development']",
            "['data science', 'sql', 'databases']",
            "['mobile', 'android', 'kotlin']",
            "['machine learning', 'tensorflow', 'keras']",
            "['python']",
            "['security', 'penetration testing', 'ethical hacking']",
            "['data science', 'machine learning', 'python']"
        ],
        'skills': [
            '["Python", "Pandas", "NumPy"]',
            '["Java", "Spring", "Hibernate"]',
            '["Python", "Flask"]',
            '["R", "Statistics"]',
            '["Python", "TensorFlow"]',
            '["JavaScript", "React"]',
            '["Python", "Selenium"]',
            '["Docker", "Kubernetes"]',
            '["Python", "Matplotlib", "Seaborn"]',
            '["Node.js", "Express"]',
            '["Python", "Scikit-learn"]',
            '["Vue.js", "CSS"]',
            '["AWS", "Azure", "GCP"]',
            '["Python", "Django"]',
            '["SQL", "PostgreSQL"]',
            '["Kotlin", "Android SDK"]',
            '["TensorFlow", "Keras"]',
            '["Python"]',
            '["Security Tools", "Metasploit"]',
            '["Python", "Scikit-learn", "Pandas"]'
        ]
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name** in `create_config_kwargs()` function
   - Default: `"job_title_current"`
   - Change to YOUR dataset's multi-valued column name
2. Run to validate field and setup environment

**What you'll see:**
- Field validation (exists, unique values, sample data)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and progress tracker ready (✓)

**Note:** Field must contain array/list-formatted data (e.g., ['item1', 'item2'])

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "job_title_current"  # Field to analyze - CUSTOMIZE THIS!
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to process: '{field_name}'")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Sample values: {list(df[field_name].unique()[:3])}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="mvf_analysis",
    description="Multi-Valued Field analysis of tags",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration parameters
- Run to execute MVF analysis
- Monitor progress bar (6 steps: setup → parse → analyze → viz → save → complete)

**Key parameters:**
- `field_name`: Multi-valued field to analyze
- `top_n=20`: Show top 20 values/combinations
- `min_frequency=1`: Include all values in dictionary
- `format_type=None`: Auto-detect array format
- `generate_visualization=True`: Create distribution charts

**What you'll see:**
- Configuration summary with all parameters
- Progress bar: validation → parsing → counting → reporting
- Completion status: "completed" (verify no errors)
- Artifact count (4-6 files expected)

**Note:** Auto-detection handles ['item'] lists and ["item"] JSON arrays

In [ ]:
# Create operation with production-style configuration
operation = MVFOperation(
    # Core parameters
    field_name=field_name,
    top_n=20,
    min_frequency=1,
    
    # Format detection
    format_type=None,  # Auto-detect
    parse_kwargs={},
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    chunk_size=10000,
    use_dask=False,
    npartitions=None,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Top N:            {operation.top_n}")
print(f"  Min frequency:    {operation.min_frequency}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing MVF analysis...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and review analysis outputs
- Compare value distributions and combinations
- Review key metrics for field complexity

**What you'll see:**
- Values dictionary (top 20 individual values, frequencies)
- Combinations dictionary (top 20 value pairs/groups, frequencies)
- Key metrics: unique values count, unique combinations, avg values/record
- Artifact list (CSV dictionaries, metrics JSON, visualizations)

**Note:** High unique combinations = High data complexity and potential privacy risk

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output files
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    print("\n" + "=" * 80)
    print("📊 RESULTS ANALYSIS")
    print("=" * 80)
    
    # Load value dictionary
    values_dict_file = [f for f in output_files if 'values_dictionary' in f.name]
    if values_dict_file:
        values_df = pd.read_csv(values_dict_file[0])
        print("\n📋 Top Values Distribution:")
        print(values_df.head(10))
    
    # Load combinations dictionary
    combinations_dict_file = [f for f in output_files if 'combinations_dictionary' in f.name]
    if combinations_dict_file:
        combinations_df = pd.read_csv(combinations_dict_file[0])
        print("\n🔗 Top Value Combinations:")
        print(combinations_df.head(10))
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:   {len(df)}")
    if values_dict_file:
        print(f"  Unique values:      {len(values_df)}")
    if combinations_dict_file:
        print(f"  Unique combos:      {len(combinations_df)}")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 MVF analysis reveals data complexity and patterns!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 2-4 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Dictionaries CSV
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance and MVF statistics
2. **Output Data**: Value and combination dictionaries
3. **Visualizations**: Charts showing distributions

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))

    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1️⃣ Exclude data_types metrics
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2️⃣ Pick latest file
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)

        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            metadata = data.get("metadata", {})
            metrics = data.get("metrics", {})

            # METADATA
            print("📋 METADATA")
            print("-" * 80)
            print(f"   Timestamp : {metadata.get('timestamp', 'N/A')}")
            print(f"   Name      : {metadata.get('name', 'N/A')}")

            op = metadata.get("operation", {})
            if op:
                print(
                    f"   Operation : {op.get('class', 'N/A')}."
                    f"{op.get('function', 'N/A')}"
                )

            # CORE METRICS
            print("\n📊 METRICS SUMMARY")
            print("-" * 80)

            def show(label, key, fmt="{}"):
                if key in metrics:
                    print(f"   {label:<28}: {fmt.format(metrics[key])}")

            show("Field Name", "field_name")
            show("Total Records", "total_records", "{:,}")
            show("Non-null Count", "non_null_count", "{:,}")
            show("Null Count", "null_count", "{:,}")
            show("Null Percentage", "null_percentage", "{:.2f}%")
            show("Empty Arrays Count", "empty_arrays_count", "{:,}")
            show("Empty Arrays %", "empty_arrays_percentage", "{:.2f}%")
            show("Unique Values", "unique_values", "{:,}")
            show("Unique Combinations", "unique_combinations", "{:,}")
            show("Avg Values / Record", "avg_values_per_record", "{:.2f}")
            show("Max Values / Record", "max_values_per_record")

            # TOP VALUES
            values = metrics.get("values_analysis")
            if values:
                print("\n📈 TOP VALUES")
                print("-" * 80)
                for i, (value, count) in enumerate(values.items(), 1):
                    if i > 10:
                        break
                    print(f"   {i:2d}. {value:<40} {count:>6,}")

            # TOP COMBINATIONS
            combos = metrics.get("combinations_analysis")
            if combos:
                print("\n🔗 TOP COMBINATIONS")
                print("-" * 80)
                for i, (combo, count) in enumerate(combos.items(), 1):
                    if i > 5:
                        break
                    combo_str = combo if len(combo) <= 60 else combo[:57] + "..."
                    print(f"   {i}. {combo_str:<60} {count:>6,}")

            # VALUE COUNT DISTRIBUTION
            dist = metrics.get("value_counts_distribution")
            if dist:
                print("\n📊 VALUE COUNT DISTRIBUTION")
                print("-" * 80)
                for k, v in sorted(dist.items(), key=lambda x: int(x[0])):
                    print(f"   {k} value(s): {v:,} record(s)")

            # NOTE
            if "note" in metrics:
                print("\n📝 NOTE")
                print("-" * 80)
                print(f"   {metrics['note']}")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. OUTPUT DATA (NEWEST)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / "output"
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob("*.csv"))

    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort newest first
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)

        # VALUES DICTIONARY
        values_files = [f for f in output_files if "values_dictionary" in f.name]
        if values_files:
            latest_values_file = values_files[0]
            mtime = datetime.fromtimestamp(latest_values_file.stat().st_mtime)

            print(f"📄 VALUES DICTIONARY")
            print(f"   File     : {latest_values_file.name}")
            print(f"   Modified : {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"   Size     : {latest_values_file.stat().st_size / 1024:.1f} KB\n")

            try:
                df = pd.read_csv(latest_values_file)

                # Normalize & sort
                df = (
                    df.rename(columns={"value": "Value"})
                      .sort_values("frequency", ascending=False)
                )
                df["percentage"] = df["percentage"].astype(float).round(2)

                print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
                print(df.head(10).to_string(index=False))

            except Exception as e:
                print(f"❌ Error reading values dictionary: {e}")

        # COMBINATIONS DICTIONARY
        combo_files = [f for f in output_files if "combinations_dictionary" in f.name]
        if combo_files:
            latest_combo_file = combo_files[0]
            mtime = datetime.fromtimestamp(latest_combo_file.stat().st_mtime)

            print("\n\n📄 COMBINATIONS DICTIONARY")
            print(f"   File     : {latest_combo_file.name}")
            print(f"   Modified : {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"   Size     : {latest_combo_file.stat().st_size / 1024:.1f} KB\n")

            try:
                df = pd.read_csv(latest_combo_file)

                df = (
                    df.rename(columns={"combination": "Combination"})
                      .sort_values("frequency", ascending=False)
                )
                df["percentage"] = df["percentage"].astype(float).round(2)

                print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
                print(df.head(10).to_string(index=False))

            except Exception as e:
                print(f"❌ Error reading combinations dictionary: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure MVF analysis with full parameters  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- MVF analysis reveals data complexity patterns
- Value distributions show most common elements
- Combination analysis identifies frequent patterns
- Auto-format detection simplifies configuration
- All artifacts saved in structured directories

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_mvf_analyzer_advanced.ipynb`** includes:
- Large dataset processing with Dask
- Parallel processing with Joblib
- Custom format parsing
- Advanced filtering and thresholds
- Performance optimization

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)